# Lesson 03 - 에이전틱 디자인 패턴

이번 수업에서는 효과적인 AI 에이전트를 구축하기 위한 세 가지 기본 디자인 패턴을 살펴봅니다:

1. **명확한 에이전트 지침** — 에이전트 행동을 안내하는 정확하고 역할을 정의하는 프롬프트 작성하기
2. **Pydantic 모델을 활용한 구조화된 출력** — 에이전트가 예측 가능하고 검증된 데이터를 반환하도록 보장하기
3. **단일 책임 에이전트** — 각각 한 가지 일을 잘 수행하는 집중된 에이전트 설계하기

각 패턴을 **여행지 추천 시스템** 시나리오에 적용해, 목적지를 추천하고, 이용 가능 여부를 확인하며, 물류를 처리할 수 있는 시스템을 단계별로 구축해 나갈 것입니다.


## 설정


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## 패턴 1: 명확한 에이전트 지침

가장 영향력 있는 패턴은 또한 가장 간단합니다: 에이전트를 위한 명확하고 상세한 지침 작성하기.

좋은 지침은 다음을 정의합니다:
- 에이전트가 <strong>누구인지</strong> (페르소나 및 어조)
- 에이전트가 **무엇을 해야 하는지** (단계별 책임)
- 에이전트가 **어떻게 행동해야 하는지** (제약 및 스타일)

아래에서는 생성하는 모든 응답을 형성하는 명확한 지침이 포함된 여행 컨시어지 에이전트를 만듭니다.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## 패턴 2: Pydantic 모델을 이용한 구조화된 출력

자유 형식 텍스트는 대화에 유용하지만, 다운스트림 시스템은 구조화된 데이터가 필요합니다.
<strong>Pydantic 모델</strong>과 <strong>도구 함수</strong>를 결합하여 우리는:

- 에이전트 출력에 대한 정확한 스키마 정의
- 응답 자동 검증
- 에이전트 결과를 애플리케이션 로직에 신뢰성 있게 통합

이를 강제하는 핵심은 에이전트를 실행할 때 `response_format`을 전달하는 것입니다. 이렇게 하면
모델이 자유 형식 텍스트 대신 검증된 `TravelRecommendations` 객체를 반환하도록 강제합니다 (`response.value`에서 사용 가능).
`get_destination_details` 도구도 타입이 지정된 `DestinationRecommendation`을 반환하여,
데이터가 끝에서 끝까지 구조화된 상태를 유지합니다.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(
    destination: Annotated[str, "The destination to look up"]
) -> DestinationRecommendation:
    """Get structured details about a vacation destination."""
    details = {
        "Barcelona": DestinationRecommendation(
            destination="Barcelona",
            available=True,
            best_season="May-Jun",
            highlights=["Beach", "Architecture", "Nightlife"],
            estimated_budget_usd=2000,
        ),
        "Tokyo": DestinationRecommendation(
            destination="Tokyo",
            available=True,
            best_season="Mar-Apr",
            highlights=["Culture", "Food", "Technology"],
            estimated_budget_usd=2500,
        ),
        "Cape Town": DestinationRecommendation(
            destination="Cape Town",
            available=False,
            best_season="Nov-Mar",
            highlights=["Nature", "Wine", "Adventure"],
            estimated_budget_usd=1800,
        ),
    }
    return details.get(
        destination,
        DestinationRecommendation(
            destination=destination,
            available=False,
            best_season="Unknown",
            highlights=[],
            estimated_budget_usd=0,
        ),
    )


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

# Passing `response_format` forces the agent to return a validated
# TravelRecommendations object instead of free-form text.
response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget",
    options={"response_format": TravelRecommendations},
)

if response and response.value:
    result: TravelRecommendations = response.value
    for rec in result.recommendations:
        status = "Available" if rec.available else "Not available"
        print(f"{rec.destination} ({status})")
        print(f"  Best season: {rec.best_season}")
        print(f"  Highlights: {', '.join(rec.highlights)}")
        print(f"  Estimated budget: ${rec.estimated_budget_usd}")
        print()
    print(f"Note: {result.personalized_note}")
else:
    print("No validated structured response was returned.")
    print(response)


## 패턴 3: 단일 책임 에이전트

복잡한 작업은 각각 단일 책임을 가진 여러 집중된 에이전트로 작업을 분할할 때 효과적입니다:

- 장소와 이용 가능성에 대해 아는 **목적지 전문가**
- 항공편, 호텔, 여행 일정 관리를 담당하는 **물류 계획자**

이는 소프트웨어 공학의 *관심사의 분리* 원칙과 일치하며 — 각 에이전트는 독립적으로 테스트, 유지 및 개선하기 쉽습니다.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## 요약

이 수업에서는 여행 추천 시나리오에 세 가지 에이전트 설계 패턴을 적용했습니다:

| 패턴 | 핵심 아이디어 | 이점 |
|---|---|---|
| **명확한 지침** | 페르소나, 책임, 제약 조건을 사전에 정의 | 일관되고 브랜드에 맞는 에이전트 행동 |
| **구조화된 출력** | Pydantic 모델을 응답 형식으로 사용 | 검증되고 기계가 읽을 수 있는 결과 |
| **단일 책임** | 각 에이전트에 하나의 집중된 작업 부여 | 테스트, 유지보수, 구성 용이 |

이러한 패턴들은 자연스럽게 조합됩니다 — 명확한 지침과 구조화된 출력을 단일 책임 에이전트 내에서 결합하여 견고하고 실제 운영에 적합한 시스템을 구축할 수 있습니다.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**면책 조항**:
이 문서는 AI 번역 서비스 [Co-op Translator](https://github.com/Azure/co-op-translator)를 사용하여 번역되었습니다. 정확성을 기하기 위해 노력하고 있으나, 자동 번역은 오류나 부정확한 부분이 있을 수 있음을 유의하시기 바랍니다. 원본 문서의 원어본이 권위 있는 자료로 간주되어야 합니다. 중요한 정보의 경우, 전문가의 인간 번역을 권장합니다. 이 번역 사용으로 인해 발생하는 오해나 잘못된 해석에 대해 당사는 책임을 지지 않습니다.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
